# Q1A | Summarize the Data

> *Summarize the data. How much data is present? What attributes/features are continuous valued? Which attributes are categorical?*

This notebook answers the question in three parts:

1. **How much data is present?** Row and column counts, the train/test split, duplicates, and where values are missing.
2. **Which attributes are continuous?** Data types, the number of distinct values, and the range of each attribute.
3. **Which attributes are categorical?** A simple classification rule applied to every column, and what that result means.

A short **Answer** section at the end pulls the three parts together.

---

#### **IMPORTANT**

This summary combines the `train` and `test` datasets to describe the full dataset. It only counts and describes the data. No modeling decisions are made here, so looking at the test data does not introduce `data snooping` bias.

# Answer Summary

- **How much data is present?** 1,893 rows and 9 columns (1 target, `Life Ladder`, and 8 features), split 82% train (1,559 rows) and 18% test (334 rows). There are no duplicate rows and no overlap between train and test. 283 values are missing, all in the train set, affecting 185 rows (11.87% of train). Perceptions of corruption and Generosity have the most gaps, and the target is never missing.
- **Which attributes are continuous?** All 9. Every column is numeric (`float64`) with hundreds of distinct values spread across a range, and no value repeats more than 17 times.
- **Which attributes are categorical?** None. No column is text or has a small set of repeated values, and the dataset does not include identifier columns like country or year.

---

### Imports

In [1]:
import pandas as pd

from Q1.consts import *

from Q1.utils import read_csv_to_df

### Fetch Data

In [2]:
train_df = read_csv_to_df(path=TRAIN_PATH)
test_df = read_csv_to_df(path=TEST_PATH)
combined_df = pd.concat([train_df, test_df], ignore_index=True)

# 1. How Much Data is Present?

In [3]:
summary = pd.DataFrame(
	{
		"row count": [train_df.shape[0], test_df.shape[0], combined_df.shape[0]],
		"% row count": [
			round(train_df.shape[0] / combined_df.shape[0] * 100, 2),
			round(test_df.shape[0] / combined_df.shape[0] * 100, 2),
			round(combined_df.shape[0] / combined_df.shape[0] * 100, 2),
		],
		"columns count": [train_df.shape[1], test_df.shape[1], combined_df.shape[1]],
		"missing values": [train_df.isnull().sum().sum(), test_df.isnull().sum().sum(), combined_df.isnull().sum().sum()],
		"rows w/ missing values": [
			train_df.isnull().any(axis=1).sum(),
			test_df.isnull().any(axis=1).sum(),
			combined_df.isnull().any(axis=1).sum()
		],
		"% rows w/ missing values": [
			round(train_df.isnull().any(axis=1).sum() / train_df.shape[0] * 100, 2),
			round(test_df.isnull().any(axis=1).sum() / test_df.shape[0] * 100, 2),
			round(combined_df.isnull().any(axis=1).sum() / combined_df.shape[0] * 100, 2),
		]
	},
	index=["train_df", "test_df", "combined_df"],
)
s = summary.T
s

,train_df,test_df,combined_df
row count,1559.00,334.00,1893.00
% row count,82.36,17.64,100.00
columns count,9.00,9.00,9.00
missing values,283.00,0.00,283.00
rows w/ missing values,185.00,0.00,185.00
% rows w/ missing values,11.87,0.00,9.77


In [4]:
features = [col for col in combined_df.columns if col != TARGET]

print(
	f"There are {s.at['row count', 'combined_df']:,.0f} rows and {s.at['columns count', 'combined_df']:.0f} columns."
)

print(
	f"Of the {s.at['columns count', 'combined_df']:.0f} columns, 1 is the target ({TARGET}) and {len(features)} are input features."
)

print(
	f"The split is {s.at['% row count', 'train_df']:.0f}% train ({s.at['row count', 'train_df']:,.0f} rows) and {s.at['% row count', 'test_df']:.0f}% test ({s.at['row count', 'test_df']:,.0f} rows)."
)

print(
	f"There are {s.at['missing values', 'train_df']:,.0f} missing training values affecting {s.at['rows w/ missing values', 'train_df']:,.0f} training rows ({s.at['% rows w/ missing values', 'train_df']:.2f}%) and {s.at['missing values', 'test_df']:,.0f} missing testing values."
)

There are 1,893 rows and 9 columns.
Of the 9 columns, 1 is the target (Life Ladder) and 8 are input features.
The split is 82% train (1,559 rows) and 18% test (334 rows).
There are 283 missing training values affecting 185 training rows (11.87%) and 0 missing testing values.


**What a row contains**

- The columns are all numeric measurements. There is **no identifier column** (no country name, year, or row ID), so each row is an anonymous observation and rows cannot be grouped or traced back to a source.
- The target, `Life Ladder`, is the value the features will be used to predict. The remaining 8 columns are the input features.

**Missing values**

- All missing values are in the **train** set. The **test** set is complete. Any missing value handling only needs to be fit on the train data.

## Duplicate Rows

In [5]:
duplicates = pd.DataFrame(
	{
		"duplicate rows": [
			train_df.duplicated().sum(),
			test_df.duplicated().sum(),
			combined_df.duplicated().sum(),
		],
	},
	index=["train_df", "test_df", "combined_df"],
)
train_test_overlap = len(train_df.merge(test_df, how="inner"))

print(f"Rows that appear in both train and test: {train_test_overlap}")
duplicates

Rows that appear in both train and test: 0


,duplicate rows
train_df,0
test_df,0
combined_df,0


There are **no duplicate rows** within either dataset, and **no row appears in both** train and test. Every row is a distinct observation, and the test set does not leak any training rows.

## Missing Values by Feature

In [6]:
missing_by_feature = pd.DataFrame({
	'Missing (train)': train_df.isna().sum(),
	'% of rows (train)': (train_df.isna().sum() / len(train_df) * 100).round(2),
	'Missing (test)': test_df.isna().sum(),
	'% of rows (test)': (test_df.isna().sum() / len(test_df) * 100).round(2),
}).sort_values('Missing (train)', ascending=False)

missing_by_feature

,Missing (train),% of rows (train),Missing (test),% of rows (test)
Perceptions of corruption,86,5.52,0,0.0
Generosity,71,4.55,0,0.0
Healthy life expectancy at birth,37,2.37,0,0.0
Log GDP per capita,26,1.67,0,0.0
Freedom to make life choices,22,1.41,0,0.0
Positive affect,17,1.09,0,0.0
Negative affect,14,0.90,0,0.0
Social support,10,0.64,0,0.0
Life Ladder,0,0.00,0,0.0


- **Perceptions of corruption** (5.52%) and **Generosity** (4.55%) have the most gaps. Together they account for 157 of the 283 missing training values.
- Every feature is missing fewer than 6% of its values, so no feature is too sparse to use.
- The target, **Life Ladder**, has no missing values, so every row can be used for training once its features are handled.

## Missing Values per Row

In [7]:
missing_per_row = train_df.isna().sum(axis=1).value_counts().sort_index()

missing_per_row_df = pd.DataFrame({
	"train rows": missing_per_row,
	"% of train rows": (missing_per_row / len(train_df) * 100).round(2),
})
missing_per_row_df.index.name = "missing values in row"

missing_per_row_df

,train rows,% of train rows
missing values in row,,
0,1374,88.13
1,130,8.34
2,27,1.73
3,23,1.48
6,5,0.32


In [8]:
sparse_rows = train_df[train_df.isna().sum(axis=1) == missing_per_row.index.max()]
present_cols = sparse_rows.columns[sparse_rows.notna().all()].tolist()

print(f"Columns present in the {len(sparse_rows)} rows missing {missing_per_row.index.max()} values: {present_cols}")

Columns present in the 5 rows missing 6 values: ['Life Ladder', 'Log GDP per capita', 'Healthy life expectancy at birth']


- 283 missing values spread over 185 rows means some rows are missing more than one value.
- Most incomplete rows (130 of 185) are missing only **1** value.
- **5 rows are missing 6 of the 8 features.** These rows only contain the target, `Log GDP per capita`, and `Healthy life expectancy at birth`. The same six columns are missing in all 5 rows.

# 2. Which Attributes are Continuous?

Two checks are needed to call an attribute continuous:

1. **Data type** rules out text labels. A column holding strings (or a mix of strings and numbers) would load as `object`.
2. **Number of distinct values** rules out numeric codes. A category stored as numbers (ex: 1, 2, 3) still loads as a number, and a column of whole numbers with any missing values loads as `float64` because `NaN` is a float. So a numeric data type alone does not prove a column is continuous.

In [9]:
info_df = pd.DataFrame({
	"dtype": combined_df.dtypes,
	"non-null values": combined_df.count(),
	"unique values": combined_df.nunique(),
	"% unique (of non-null)": (combined_df.nunique() / combined_df.count() * 100).round(2),
	"most frequent value count": combined_df.apply(lambda col: col.value_counts().iloc[0]),
})

info_df

,dtype,non-null values,unique values,% unique (of non-null),most frequent value count
Life Ladder,float64,1893,1519,80.24,5
Log GDP per capita,float64,1867,1476,79.06,4
Social support,float64,1883,452,24.00,15
Healthy life expectancy at birth,float64,1856,825,44.45,15
Freedom to make life choices,float64,1871,533,28.49,10
Generosity,float64,1822,602,33.04,13
Perceptions of corruption,float64,1807,566,31.32,14
Positive affect,float64,1876,430,22.92,12
Negative affect,float64,1879,372,19.80,17


- Every column is `float64`, so there are no text labels.
- Every column has **hundreds of distinct values** (372 to 1,519), far more than a set of category labels.
- No single value dominates. The most common value in any column repeats at most 17 times out of about 1,800 rows.

**Why some columns are only 20% to 30% unique**

Every value is recorded to at most 3 decimal places. The features that fall between 0 and 1 can only take about 1,000 distinct values at that precision, and their observed ranges are narrower still. For example, `Negative affect` ranges from 0.083 to 0.705, which allows only 623 possible values for 1,879 rows. Repeated values are therefore forced by rounding. They do not mean the values are category levels.

## Value Ranges

In [10]:
feature_dictionary = pd.DataFrame({
	"role": ["target" if col == TARGET else "feature" for col in combined_df.columns],
	"min": combined_df.min(),
	"max": combined_df.max(),
	"mean": combined_df.mean().round(3),
	"std": combined_df.std().round(3),
}, index=combined_df.columns)

feature_dictionary

,role,min,max,mean,std
Life Ladder,target,2.375,8.019,5.461,1.124
Log GDP per capita,feature,6.635,11.648,9.356,1.156
Social support,feature,0.290,0.987,0.812,0.119
Healthy life expectancy at birth,feature,32.300,77.100,63.337,7.546
Freedom to make life choices,feature,0.258,0.985,0.741,0.142
Generosity,feature,-0.335,0.698,-0.000,0.162
Perceptions of corruption,feature,0.035,0.983,0.748,0.187
Positive affect,feature,0.322,0.944,0.709,0.107
Negative affect,feature,0.083,0.705,0.269,0.085


- Every attribute takes values across a **range**, not from a fixed list.
- Five features (`Social support`, `Freedom to make life choices`, `Perceptions of corruption`, `Positive affect`, `Negative affect`) fall between 0 and 1.
- `Healthy life expectancy at birth` ranges from 32.3 to 77.1.
- `Generosity` has a mean of about 0 and is the only attribute with negative values (min -0.335).
- The features use very different ranges (0 to 1 vs. 32.3 to 77.1), which is worth noting for any model that is sensitive to feature scale.

**All 9 attributes (the target and all 8 features) are continuous.**

# 3. Which Attributes are Categorical?

To make the classification explicit, each column is tested with a simple rule:

> A column is **categorical** if its data type is `object` / `category`, or if it has 20 or fewer distinct values. Otherwise it is **continuous**.

The threshold of 20 is generous. Every column here has at least 372 distinct values, so the result does not depend on the exact cutoff.

In [11]:
CATEGORICAL_MAX_UNIQUE = 20

classification = pd.DataFrame({
	"dtype": combined_df.dtypes,
	"unique values": combined_df.nunique(),
})
classification["text dtype"] = [
	pd.api.types.is_object_dtype(dtype) or isinstance(dtype, pd.CategoricalDtype)
	for dtype in classification["dtype"]
]
classification["<= 20 unique"] = classification["unique values"] <= CATEGORICAL_MAX_UNIQUE
classification["classification"] = [
	"categorical" if is_text or is_few else "continuous"
	for is_text, is_few in zip(classification["text dtype"], classification["<= 20 unique"])
]

print(classification["classification"].value_counts().to_string())
classification

classification
continuous    9


,dtype,unique values,text dtype,<= 20 unique,classification
Life Ladder,float64,1519,False,False,continuous
Log GDP per capita,float64,1476,False,False,continuous
Social support,float64,452,False,False,continuous
Healthy life expectancy at birth,float64,825,False,False,continuous
Freedom to make life choices,float64,533,False,False,continuous
Generosity,float64,602,False,False,continuous
Perceptions of corruption,float64,566,False,False,continuous
Positive affect,float64,430,False,False,continuous
Negative affect,float64,372,False,False,continuous


**No attributes are categorical.**

- No column fails either check: none are text, and none have a small set of repeated values.
- The dataset does not include columns such as **country**, **region**, or **year**.
- A categorical feature could still be created later (ex: binning `Log GDP per capita` into income groups), but none exist in the data as given.